# Age structure per municipality: young and older residents

This notebook maps the share of young residents (0 to 25 years) and older residents (65 years and over) in all Dutch municipalities, and saves the age data for the analysis as `data/processed/age_per_municipality.csv`.

**Required files:**
- `data/raw/jongeren_per_gemeente_2026.csv`: share of residents aged 0 to 25 per municipality, 2026 (% of total population)
- `data/raw/ouderen_per_gemeente_2026.csv`: share of residents aged 65 and over per municipality, 2026 (% of total population)
- `data/raw/gemeenten_2024.gpkg`: municipal boundaries from CBS/PDOK, 2024 municipal division

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 1. Load the data

Both files use `;` as separator and a comma as decimal sign. We keep only the total columns and give them short names.

In [ ]:
young = pd.read_csv("../data/raw/jongeren_per_gemeente_2026.csv", sep=";", decimal=",", encoding="utf-8-sig")
old = pd.read_csv("../data/raw/ouderen_per_gemeente_2026.csv", sep=";", decimal=",", encoding="utf-8-sig")

young = young[["Gemeente", "Totaal, 0 tot 25"]].rename(columns={"Gemeente": "municipality", "Totaal, 0 tot 25": "pct_young_0_25"})
old = old[["Gemeente", "Totaal, 65 jaar of ouder"]].rename(columns={"Gemeente": "municipality", "Totaal, 65 jaar of ouder": "pct_older_65plus"})

age = young.merge(old, on="municipality", how="inner")
print("Number of municipalities:", len(age))
display(age.head())

## 2. Load municipal boundaries

In [ ]:
layer = "gemeente_gegeneraliseerd"
municipalities = gpd.read_file("../data/raw/gemeenten_2024.gpkg", layer=layer)

print("Number of municipalities on the map:", len(municipalities))

## 3. Harmonise municipality names

The age files add "(gemeente)" to three municipality names; the boundary file does not.

In [ ]:
name_mapping_map = {
    "'s-Gravenhage (gemeente)": "'s-Gravenhage",
    "Groningen (gemeente)": "Groningen",
    "Utrecht (gemeente)": "Utrecht",
}
age_map = age.copy()
age_map["municipality"] = age_map["municipality"].replace(name_mapping_map)

municipalities["municipality"] = municipalities["statnaam"]

# Check: both lines should print set()
print("Only on the map:", set(municipalities["municipality"]) - set(age_map["municipality"]))
print("Only in the data:", set(age_map["municipality"]) - set(municipalities["municipality"]))

## 4. Join the data to the map

In [ ]:
age_gdf = municipalities.merge(age_map, on="municipality", how="left")
print("Municipalities without data:", age_gdf["pct_young_0_25"].isna().sum())

## 5. Create the maps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

age_gdf.plot(
    column="pct_young_0_25",
    cmap="Greens",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Residents aged 0-25 (% of population)", "shrink": 0.6},
    ax=axes[0],
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
axes[0].set_title("Share of young residents (aged 0-25), 2026", fontsize=14)
axes[0].axis("off")

age_gdf.plot(
    column="pct_older_65plus",
    cmap="Purples",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Residents aged 65+ (% of population)", "shrink": 0.6},
    ax=axes[1],
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
axes[1].set_title("Share of older residents (aged 65+), 2026", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/map_age_per_municipality.png", dpi=200)
plt.show()

## 6. Save for the analysis

For the regression, the age data will later be merged with the charging point data. We therefore use the same municipality names as the charging point file (the same mapping as in notebook 1), so both files can be merged on `municipality`.

In [ ]:
name_mapping_charging = {
    "'s-Gravenhage (gemeente)": "Den Haag",
    "Groningen (gemeente)": "Groningen",
    "Utrecht (gemeente)": "Utrecht",
    "Beek (L.)": "Beek",
    "Stein (L.)": "Stein",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
}
age_save = age.copy()
age_save["municipality"] = age_save["municipality"].replace(name_mapping_charging)

# Check: does every municipality match the charging-point data?
charging = pd.read_csv("../data/processed/chargingpoints_households_income.csv")
print("Not matched with charging-point data:", set(age_save["municipality"]) - set(charging["municipality"]))

age_save.to_csv("../data/processed/age_per_municipality.csv", index=False)
print("Saved:", len(age_save), "municipalities")